In [121]:
import pandas as pd

In [122]:
df = pd.read_csv("data_with_elo.csv", index_col=0)

In [123]:
df["gameDateTimeEst"] = pd.to_datetime(df["gameDateTimeEst"])

In [124]:
df["win"] = df["win"].astype(bool)

In [125]:
df["game_date"] = df["gameDateTimeEst"].dt.date
df["last_game_played"] = df.groupby("teamName")["game_date"].shift(1)

In [126]:
df['game_date'] = pd.to_datetime(df['game_date'])
df['last_game_played'] = pd.to_datetime(df['last_game_played'])

In [127]:
df["days_rest"] = (df["game_date"] - df["last_game_played"]).dt.days
df["is_B2B"] = df["days_rest"] == 1

In [128]:
df = df.sort_values(by=["gameDateTimeEst", "gameId"])

In [129]:
rolling_features = ['possessions', 'eFG', 'TO%',
       'OREB%', 'FTR', 'off_rating', 'def_rating', 'net_rating']


for feature in rolling_features:
       df[f"{feature}_rolling"] = df.groupby('teamName')[f"{feature}"].transform(lambda x: x.ewm(span=10).mean())

In [130]:
ewma_cols = [f"{f}_rolling" for f in rolling_features]
df[ewma_cols] = df.groupby('teamName')[ewma_cols].shift(1)

In [131]:
df = df.sort_values(by=["gameDateTimeEst"])

In [132]:
df = df.drop(columns=["points","opponentScore", "teamId", "season"])
df = df.drop(columns=rolling_features)

In [133]:
home_df = df[df["home"] == 1]
away_df = df[df["home"] == 0]

game_df = pd.merge(home_df, away_df, on='gameId', suffixes=('_home', '_away'))

In [134]:
game_df.columns

Index(['gameId', 'gameDateTimeEst_home', 'teamName_home', 'home_home',
       'win_home', 'pre_game_elo_home', 'game_date_home',
       'last_game_played_home', 'days_rest_home', 'is_B2B_home',
       'possessions_rolling_home', 'eFG_rolling_home', 'TO%_rolling_home',
       'OREB%_rolling_home', 'FTR_rolling_home', 'off_rating_rolling_home',
       'def_rating_rolling_home', 'net_rating_rolling_home',
       'gameDateTimeEst_away', 'teamName_away', 'home_away', 'win_away',
       'pre_game_elo_away', 'game_date_away', 'last_game_played_away',
       'days_rest_away', 'is_B2B_away', 'possessions_rolling_away',
       'eFG_rolling_away', 'TO%_rolling_away', 'OREB%_rolling_away',
       'FTR_rolling_away', 'off_rating_rolling_away',
       'def_rating_rolling_away', 'net_rating_rolling_away'],
      dtype='object')

In [135]:
game_df

,gameId,gameDateTimeEst_home,teamName_home,home_home,win_home,pre_game_elo_home,game_date_home,last_game_played_home,days_rest_home,is_B2B_home,...,days_rest_away,is_B2B_away,possessions_rolling_away,eFG_rolling_away,TO%_rolling_away,OREB%_rolling_away,FTR_rolling_away,off_rating_rolling_away,def_rating_rolling_away,net_rating_rolling_away
0,28600009,1986-10-31 20:00:00,Suns,1,True,1444.3523,1986-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,28600008,1986-10-31 20:00:00,Mavericks,1,True,1547.2805,1986-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,28600007,1986-10-31 20:00:00,Nuggets,1,True,1514.6084,1986-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,28600006,1986-10-31 20:00:00,Pistons,1,False,1532.8929,1986-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,28600005,1986-10-31 20:00:00,Kings,1,True,1461.1960,1986-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51330,42500207,2026-05-17 20:00:00,Pistons,1,False,1691.9800,2026-05-17,2026-05-15,2.0,False,...,2.0,False,93.219258,0.532566,0.171521,0.302661,0.369080,116.298580,118.802941,-2.504362
51331,42500311,2026-05-18 20:30:00,Thunder,1,False,1767.2500,2026-05-18,2026-05-11,7.0,False,...,3.0,False,96.995878,0.575412,0.138663,0.280188,0.304428,125.068780,108.302345,16.766435
51332,42500301,2026-05-19 20:00:00,Knicks,1,True,1741.2900,2026-05-19,2026-05-10,9.0,False,...,2.0,False,93.962230,0.539479,0.160889,0.304197,0.396093,118.509976,114.766564,3.743412
51333,42500312,2026-05-20 20:30:00,Thunder,1,True,1756.8900,2026-05-20,2026-05-18,2.0,False,...,2.0,False,99.391100,0.560754,0.148109,0.288533,0.304001,122.463248,107.590012,14.873236


In [136]:
difference_features = ['days_rest', 'possessions_rolling', 'eFG_rolling', 'TO%_rolling',
       'OREB%_rolling', 'FTR_rolling', 'off_rating_rolling', 'def_rating_rolling', 'net_rating_rolling', "pre_game_elo"]

for features in difference_features:
    game_df[f"{features}_diff"] = game_df[f"{features}_home"] - game_df[f"{features}_away"]

In [137]:
game_df = game_df.rename(columns={"gameDateTimeEst_home":"game_date"})
final_df = game_df[['game_date','pre_game_elo_home', 'is_B2B_home',
       'pre_game_elo_away', 'is_B2B_away', 'pre_game_elo_diff','days_rest_diff','possessions_rolling_diff', 'eFG_rolling_diff',
       'TO%_rolling_diff', 'OREB%_rolling_diff', 'FTR_rolling_diff',
       'off_rating_rolling_diff', 'def_rating_rolling_diff',
       'net_rating_rolling_diff', 'win_home']]

In [138]:
final_df.columns

Index(['game_date', 'pre_game_elo_home', 'is_B2B_home', 'pre_game_elo_away',
       'is_B2B_away', 'pre_game_elo_diff', 'days_rest_diff',
       'possessions_rolling_diff', 'eFG_rolling_diff', 'TO%_rolling_diff',
       'OREB%_rolling_diff', 'FTR_rolling_diff', 'off_rating_rolling_diff',
       'def_rating_rolling_diff', 'net_rating_rolling_diff', 'win_home'],
      dtype='object')

In [139]:
final_df = final_df.dropna()

In [140]:
final_df = final_df.reset_index(drop=True)

In [141]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51315 entries, 0 to 51314
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   game_date                 51315 non-null  datetime64[ns]
 1   pre_game_elo_home         51315 non-null  float64       
 2   is_B2B_home               51315 non-null  bool          
 3   pre_game_elo_away         51315 non-null  float64       
 4   is_B2B_away               51315 non-null  bool          
 5   pre_game_elo_diff         51315 non-null  float64       
 6   days_rest_diff            51315 non-null  float64       
 7   possessions_rolling_diff  51315 non-null  float64       
 8   eFG_rolling_diff          51315 non-null  float64       
 9   TO%_rolling_diff          51315 non-null  float64       
 10  OREB%_rolling_diff        51315 non-null  float64       
 11  FTR_rolling_diff          51315 non-null  float64       
 12  off_rating_rolling

In [142]:
final_df.shape

(51315, 16)

In [143]:
final_df.to_csv("/Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/data/final_dataset.csv")